# Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/spinal-bone-feature-detection
!ls

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/.shortcut-targets-by-id/1oG2uzFRqi9wz763XVJclWvhPaLhUUFFX/spinal-bone-feature-detection
backup	     datasets	  kfolds     models.py	  results      utils.py
config.yaml  experiments  loader.py  __pycache__  trainers.py  weights


In [2]:
!unzip -q ./datasets/ultrasound.zip -d /tmp/ultrasound
!pip install -q torchmetrics[detection]
# !pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 46.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.optim as optim
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

import gc
from models import custom_resnet18_backbone
from trainers import FasterRCNNTrainer
from loader import ScoliosisDataset, get_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits_path = 'kfolds/splits4.yaml'
best_model_path = 'weights/faster_rcnn4.pth'
!rm -rf results
%load_ext tensorboard

# Training

In [3]:
train_loader = get_loader('train', splits_path=splits_path, batch_size=32, use_bg_class=True)
val_loader = get_loader('val', splits_path=splits_path, batch_size=32, use_bg_class=True)
model = FasterRCNN(
    custom_resnet18_backbone(), num_classes=3, # 2 classes (Thoracic, Lumbar) + background
    rpn_anchor_generator=AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-4)

[kfolds/splits4.yaml] Train dataset: 248 samples
[kfolds/splits4.yaml] Val dataset: 61 samples


In [4]:
temp_dataset = ScoliosisDataset(split='train', config_path='config.yaml', splits_path=splits_path, use_bg_class=True)
dataset_mean, dataset_std = temp_dataset.means, temp_dataset.stds

# Update the model's internal transform with the correct mean and std
# Access the transform through model.transform
model.transform.image_mean = dataset_mean.tolist()
model.transform.image_std = dataset_std.tolist()

In [5]:
%%time
gc.collect()
torch.cuda.empty_cache()
trainer = FasterRCNNTrainer(model, train_loader, val_loader, num_epochs=1000, optimizer=optimizer, best_model_path=best_model_path)
trainer.train()

[EPOCH 1/1000] Train Loss: 1.7250 - Val Loss: 1.3979
[EPOCH 4/1000] Train Loss: 1.2914 - Val Loss: 1.3649
[EPOCH 6/1000] Train Loss: 1.2521 - Val Loss: 1.2489
[EPOCH 16/1000] Train Loss: 1.1656 - Val Loss: 1.2308
[EPOCH 17/1000] Train Loss: 1.1506 - Val Loss: 1.2265
[EPOCH 18/1000] Train Loss: 1.0999 - Val Loss: 1.1563
[EPOCH 19/1000] Train Loss: 1.1040 - Val Loss: 1.1352
[EPOCH 24/1000] Train Loss: 1.0708 - Val Loss: 1.0830
[EPOCH 25/1000] Train Loss: 1.0580 - Val Loss: 1.0715
[EPOCH 26/1000] Train Loss: 1.0338 - Val Loss: 1.0503
[EPOCH 27/1000] Train Loss: 1.0431 - Val Loss: 1.0382
[EPOCH 28/1000] Train Loss: 1.0269 - Val Loss: 1.0299
[EPOCH 29/1000] Train Loss: 1.0369 - Val Loss: 1.0191
[EPOCH 34/1000] Train Loss: 1.0382 - Val Loss: 1.0053
[EPOCH 35/1000] Train Loss: 0.9792 - Val Loss: 0.9942
[EPOCH 36/1000] Train Loss: 0.9815 - Val Loss: 0.9829
[EPOCH 39/1000] Train Loss: 0.9706 - Val Loss: 0.9699
[EPOCH 40/1000] Train Loss: 0.9669 - Val Loss: 0.9684
[EPOCH 41/1000] Train Loss: 0.9

# mAP Evaluation

In [ ]:
%tensorboard --logdir results

In [7]:
checkpoint = torch.load(best_model_path)
trainer.model.load_state_dict(checkpoint)
mAP_preds, mAP_targets, results = trainer.evaluate()
results

{'map': tensor(0.4974),
 'map_50': tensor(0.9279),
 'map_75': tensor(0.4830),
 'map_small': tensor(0.4669),
 'map_medium': tensor(0.5137),
 'map_large': tensor(-1.),
 'mar_1': tensor(0.0706),
 'mar_10': tensor(0.4518),
 'mar_100': tensor(0.5756),
 'mar_small': tensor(0.5259),
 'mar_medium': tensor(0.6040),
 'mar_large': tensor(-1.),
 'map_per_class': tensor([0.4839, 0.5110]),
 'mar_100_per_class': tensor([0.5600, 0.5912]),
 'classes': tensor([1, 2], dtype=torch.int32)}